# Missing-road finder
Predicts roads on every Yaou tile with a model that never trained on that tile, then flags roads the model sees that OSM does not have.
Runtime > Change runtime type > T4 GPU. Run the cells one by one with Shift + Enter. Total time is about 15 minutes.

In [ ]:
# 1. Install and upload Dataset.zip (the zip with images/ and masks/)
!pip -q install segmentation-models-pytorch rasterio
import os, glob, numpy as np, torch, random, rasterio
from rasterio.merge import merge
from scipy import ndimage
import segmentation_models_pytorch as smp
from google.colab import files
torch.manual_seed(0); np.random.seed(0); random.seed(0)
dev = "cuda"
files.upload()
!unzip -q -o "$(ls *.zip | grep -i dataset | head -1)" -d data
root = "data" if os.path.isdir("data/images") else glob.glob("data/*/images")[0].rsplit("/images", 1)[0]
print("root:", root)

In [ ]:
# 2. Load every tile with its map coordinates
names = sorted(os.path.basename(p) for p in glob.glob(f"{root}/images/*.tif"))
names = [n for n in names if os.path.exists(f"{root}/masks/{n}")]
tiles = {}
for n in names:
    with rasterio.open(f"{root}/images/{n}") as s:
        img = s.read()[:3].transpose(1, 2, 0).astype(np.uint8); prof = s.profile
    with rasterio.open(f"{root}/masks/{n}") as s:
        m = (s.read(1) > 0).astype(np.float32)
    gx = int(n.rsplit(".", 1)[0].split("_")[-2])
    tiles[n] = dict(img=img, mask=m, prof=prof, gx=gx)
cols = sorted({t["gx"] for t in tiles.values()}); half = cols[len(cols) // 2]
for t in tiles.values(): t["fold"] = 0 if t["gx"] < half else 1
print(len(tiles), "tiles | west half:", sum(t["fold"] == 0 for t in tiles.values()),
      "| east half:", sum(t["fold"] == 1 for t in tiles.values()))

In [ ]:
# 3. Train two models: one on the west half, one on the east half (about 4 min each)
MEAN = np.array([0.485, 0.456, 0.406]); STD = np.array([0.229, 0.224, 0.225])
def to_x(img): return torch.tensor(((img / 255.0 - MEAN) / STD).transpose(2, 0, 1).copy(), dtype=torch.float32)
class DS(torch.utils.data.Dataset):
    def __init__(self, ts): self.ts = ts
    def __len__(self): return len(self.ts)
    def __getitem__(self, i):
        img, m = self.ts[i]["img"], self.ts[i]["mask"]
        k = np.random.randint(4); img, m = np.rot90(img, k), np.rot90(m, k)
        if np.random.rand() < .5: img, m = img[:, ::-1], m[:, ::-1]
        return to_x(img), torch.tensor(m.copy())[None]
dice = smp.losses.DiceLoss("binary"); bce = torch.nn.BCEWithLogitsLoss()
models = {}
for fold in [0, 1]:
    tr = [t for t in tiles.values() if t["fold"] != fold and t["mask"].mean() > 0.005]
    dl = torch.utils.data.DataLoader(DS(tr), batch_size=4, shuffle=True)
    model = smp.DeepLabV3Plus("resnet34", encoder_weights="imagenet", classes=1).to(dev)
    opt = torch.optim.AdamW(model.parameters(), 1e-4)
    for ep in range(20):
        model.train(); tot = 0
        for x, y in dl:
            x, y = x.to(dev), y.to(dev); out = model(x); loss = dice(out, y) + bce(out, y)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        print(f"model for {'west' if fold == 0 else 'east'} half | ep {ep+1:2d} loss {tot/len(dl):.3f}")
    models[fold] = model

In [ ]:
# 4. Predict every tile with the model that never saw it, then flag roads OSM is missing
os.makedirs("out/pred", exist_ok=True); os.makedirs("out/missing", exist_ok=True)
for n, t in tiles.items():
    model = models[t["fold"]]; model.eval()
    with torch.no_grad():
        p = torch.sigmoid(model(to_x(t["img"])[None].to(dev)))[0, 0].cpu().numpy() > 0.5
    near_osm = ndimage.binary_dilation(t["mask"] > 0, iterations=6)          # within 3 m of an OSM road
    miss = p & ~near_osm
    lab, k = ndimage.label(miss)
    sizes = ndimage.sum(miss, lab, range(1, k + 1))
    miss = np.isin(lab, [i + 1 for i, s in enumerate(sizes) if s >= 100])    # keep pieces of at least 25 m²
    prof = t["prof"].copy(); prof.update(count=1, dtype="uint8", nodata=0, compress="deflate")
    for folder, arr in [("pred", p), ("missing", miss)]:
        with rasterio.open(f"out/{folder}/{n}", "w", **prof) as d: d.write((arr * 255).astype(np.uint8), 1)
for folder in ["pred", "missing"]:
    arr, tr = merge(sorted(glob.glob(f"out/{folder}/*.tif")))
    prof = t["prof"].copy(); prof.update(count=1, dtype="uint8", nodata=0, compress="deflate",
                                          height=arr.shape[1], width=arr.shape[2], transform=tr)
    with rasterio.open(f"out/{folder}_mosaic.tif", "w", **prof) as d: d.write(arr[0], 1)
print("done")

In [ ]:
# 5. Turn flagged areas into map shapes and count them
import geopandas as gpd
from rasterio.features import shapes
from shapely.geometry import shape
with rasterio.open("out/missing_mosaic.tif") as s:
    a = s.read(1); geoms = [shape(g) for g, v in shapes(a, mask=a > 0, transform=s.transform) if v > 0]; crs = s.crs
gdf = gpd.GeoDataFrame({"area_m2": [g.area for g in geoms]}, geometry=geoms, crs=crs)
gdf["approx_length_m"] = (gdf.area_m2 / 3).round()
gdf = gdf.sort_values("area_m2", ascending=False).reset_index(drop=True); gdf["flag_id"] = gdf.index + 1
gdf.to_file("out/possible_missing_roads.gpkg", driver="GPKG")
print(f"{len(gdf)} possible missing road segments, about {gdf.approx_length_m.sum()/1000:.1f} km in total")
print(gdf.head(10)[["flag_id", "area_m2", "approx_length_m"]])

In [ ]:
# 6. Download the results
!cd out && zip -q ../map_layers.zip pred_mosaic.tif missing_mosaic.tif possible_missing_roads.gpkg
files.download("map_layers.zip")